# 2. ETF Screening and Selection

## Overview
This notebook implements the ETF screening and selection process, focusing on:
1. Filtering for distributing ETFs only
2. Platform availability verification
3. Initial ranking based on TER and dividends
4. Risk-adjusted returns analysis

## Setup and Data Loading

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import os
import json
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import requests
from scipy import stats

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import get_region_category_from_filename, get_asset_class_from_filename
from etf_utils.metrics import calculate_annualized_volatility
from etf_utils.platform_check import check_etf_availability

provider = DataProvider()

## Platform Availability Check
First, we'll define a function to check if ETFs are available on our chosen platform (InvestEngine):

In [ ]:
# check_etf_availability is imported from etf_utils.platform_check
# Example usage:
# check_etf_availability("VEVE")  # returns True/False

In [ ]:
# Calculate benchmark returns and volatility for 2024
# Using DataProvider (yfinance by default, no API key needed)

benchmark_year_start = "2024-01-01"
benchmark_year_end = "2024-12-31"

eq_benchmark_symbol = "VEVE"  # VEVE.L on yfinance
bnd_benchmark_symbol = "SAAA"  # SAAA.L on yfinance

eq_prices = provider.get_historical_prices(eq_benchmark_symbol)
bnd_prices = provider.get_historical_prices(bnd_benchmark_symbol)

eq_year = eq_prices.loc[benchmark_year_start:benchmark_year_end]["close"]
bnd_year = bnd_prices.loc[benchmark_year_start:benchmark_year_end]["close"]

eq_benchmark_return = round((eq_year.iloc[-1] - eq_year.iloc[0]) / eq_year.iloc[0] * 100, 2)
bnd_benchmark_return = round((bnd_year.iloc[-1] - bnd_year.iloc[0]) / bnd_year.iloc[0] * 100, 2)
eq_benchmark_volatility = round(calculate_annualized_volatility(eq_year) * 100, 2)
bnd_benchmark_volatility = round(calculate_annualized_volatility(bnd_year) * 100, 2)

eq_sharpe_ratio = round(eq_benchmark_return / eq_benchmark_volatility, 2)
bond_sharpe_ratio = round(bnd_benchmark_return / bnd_benchmark_volatility, 2)

print(f"2024 VEVE return: {eq_benchmark_return}%")
print(f"2024 SAAA return: {bnd_benchmark_return}%")
print(f"2024 VEVE volatility: {eq_benchmark_volatility}%")
print(f"2024 SAAA volatility: {bnd_benchmark_volatility}%")
print(f"2024 VEVE Sharpe ratio: {eq_sharpe_ratio}")
print(f"2024 SAAA Sharpe ratio: {bond_sharpe_ratio}")

In [ ]:
benchmark_etfs = [
    # Structure: [Asset Class, Region, Country, Primary Ticker, Description]
    
    # Equity Benchmarks - Developed Markets
    ['Equity', 'Developed_AmericasandUK', 'United Kingdom', 'ISF.LON', 'iShares Core FTSE 100 UCITS ETF GBP (Dist)'],
    ['Equity', 'Developed_AmericasandUK', 'United States', 'SPY', 'SPDR S&P 500 ETF Trust'],
    ['Equity', 'Developed_AmericasandUK', 'Canada', 'ZCN.TRT', 'BMO S&P/TSX Capped Composite'],    
    ['Equity', 'Developed_EMEA', 'Germany', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'France', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Italy', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Spain', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Netherlands', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Switzerland', 'CSWG.LON', 'Amundi Index Solutions - Amundi MSCI Switzerland'],    
    ['Equity', 'Developed_APAC', 'Japan', 'XDJP.LON', 'Xtrackers Nikkei 225 UCITS ETF 1D'],
    ['Equity', 'Developed_APAC', 'Australia', 'SAUS.LON', 'iShares MSCI Australia UCITS ETF'],    
    # Equity Benchmarks - Emerging Markets
    ['Equity', 'Emerging_Americas', 'Brazil', 'IBZL.LON', 'iShares MSCI Brazil UCITS ETF'],
    ['Equity', 'Emerging_Americas', 'Mexico', 'XMEX.LON', 'iShares MSCI Mexico Capped ETF'],    
    ['Equity', 'Emerging_APACandEMEA', 'China', 'ASHR', 'XTRACKERS HARVEST CSI 300 CHINA A-SHARES ETF'],
    ['Equity', 'Emerging_APACandEMEA', 'India', 'XNIF.LON', 'Xtrackers Nifty 50 Swap UCITS ETF 1C'],
    ['Equity', 'Emerging_APACandEMEA', 'South Korea', 'EWY', 'iShares MSCI South Korea ETF'],    
]

benchmark_df = pd.DataFrame(
    benchmark_etfs, 
    columns=['asset_class', 'region', 'country', 'benchmark_ticker', 'benchmark_description']
)

# Sort the DataFrame
benchmark_df = benchmark_df.sort_values(['asset_class', 'region', 'country'])

# Calculate 2024 returns for each benchmark ETF using DataProvider
benchmark_df['benchmark_2024_Return'] = benchmark_df['benchmark_ticker'].apply(
    lambda sym: round(provider.get_benchmark_period_return(sym, "2024-01-01", "2024-12-31") * 100, 2)
)
benchmark_df

In [5]:
benchmark_df

,asset_class,region,country,benchmark_ticker,benchmark_description,benchmark_2024_Return
10,Equity,Developed_APAC,Australia,SAUS.LON,iShares MSCI Australia UCITS ETF,2.81
9,Equity,Developed_APAC,Japan,XDJP.LON,Xtrackers Nikkei 225 UCITS ETF 1D,9.43
2,Equity,Developed_AmericasandUK,Canada,ZCN.TRT,BMO S&P/TSX Capped Composite,22.13
0,Equity,Developed_AmericasandUK,United Kingdom,ISF.LON,iShares Core FTSE 100 UCITS ETF GBP (Dist),9.52
1,Equity,Developed_AmericasandUK,United States,SPY,SPDR S&P 500 ETF Trust,25.59
4,Equity,Developed_EMEA,France,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74
3,Equity,Developed_EMEA,Germany,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74
5,Equity,Developed_EMEA,Italy,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74
7,Equity,Developed_EMEA,Netherlands,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74
6,Equity,Developed_EMEA,Spain,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74


### Fetch previous day close price from alpha vantage

In [ ]:
# get_latest_price is now available via provider.get_latest_price(symbol)
# Example usage:
# date_str, price = provider.get_latest_price("VEVE")  # returns (date, price) tuple

## ETF Screening Process
Now proceed with screening and categorization using pre-determined weights by country

In [ ]:
# get_region_category_from_filename and get_asset_class_from_filename
# are imported from etf_utils.data_io

main_df_equities = pd.DataFrame()

for filepath in sorted(DATA_RAW.glob("justetf_class-equity*.csv")):
    csvl = filepath.name
    try:
        asset_class = get_asset_class_from_filename(csvl)
        region_category = get_region_category_from_filename(csvl)

        if not asset_class or region_category == 'Unknown' or asset_class != 'equity':
            print(f'Skipping {csvl}')
            continue

        print(f'Processing {asset_class} file for {region_category}: {csvl}')

        try:
            csv_df = pd.read_csv(filepath)
            if csv_df.empty:
                print(f'Empty file: {csvl}')
                continue
        except pd.errors.EmptyDataError:
            print(f'Empty file: {csvl}')
            continue

        csv_df['asset_class'] = asset_class
        csv_df['region'] = csv_df['region'].fillna('N/A')
        csv_df['country'] = csv_df['country'].fillna('N/A')

        distributing_df = csv_df.copy()
        distributing_df = distributing_df[distributing_df['dividends'] == 'Distributing']
        distributing_df = distributing_df[distributing_df['size'] > 100]
        distributing_df = distributing_df[distributing_df['hedged'] == False]

        include_tickers = ['IBZL']
        distributing_df = distributing_df[
            (distributing_df['ter'] < 0.5) | (distributing_df['ticker'].isin(include_tickers))
        ]

        # Tickers to exclude (re-evaluate with yfinance — some may now be available)
        remove_tickers = ['XUCN', 'LYXIB', 'C001', 'SW2CHA', 'XSMI', 'SJPD', 'XESD', 'C024']
        distributing_df = distributing_df[~distributing_df['ticker'].isin(remove_tickers)]

        hy_df = (distributing_df.copy()
                 .sort_values(by=['last_year_dividends'], ascending=False)
                 .drop_duplicates(subset=['region_category']))
        hy_df['intra_region_category'] = 'High Yield'

        benchmark_distributing_df = distributing_df.copy()[
            ~distributing_df['country'].isin(hy_df['country'])
        ]
        benchmark_distributing_df = pd.merge(
            benchmark_distributing_df,
            benchmark_df[['country', 'benchmark_ticker', 'benchmark_description', 'benchmark_2024_Return']],
            on='country', how='left'
        )
        benchmark_distributing_df["beta"] = (
            benchmark_distributing_df["2024"] / benchmark_distributing_df["benchmark_2024_Return"]
        )

        # Save intermediate benchmark data
        intermediate_path = DATA_INTERMEDIATE / f'benchmark_distributing_df_{csvl}'
        benchmark_distributing_df.to_csv(intermediate_path, index=False)

        benchmark_distributing_df = benchmark_distributing_df[benchmark_distributing_df['beta'] >= 1]
        beta_df = (benchmark_distributing_df.copy()
                   .sort_values(by=['ter', 'beta'], ascending=[True, False])
                   .drop_duplicates(subset=['region_category']))
        beta_df['intra_region_category'] = 'Beta'

        main_df_equities = pd.concat([main_df_equities, hy_df, beta_df], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')

# Check InvestEngine availability
main_df_equities['available_on_investengine'] = (
    main_df_equities['ticker'].apply(check_etf_availability)
)

# Fetch latest close price via DataProvider (yfinance, no API key)
def _get_price(ticker):
    try:
        return provider.get_latest_price(ticker)
    except Exception as e:
        print(f"Price error for {ticker}: {e}")
        return None, None

main_df_equities[['yday_close_price_date', 'yday_close_price']] = (
    main_df_equities['ticker'].apply(_get_price).to_list()
)

main_df_equities.to_csv(DATA_INTERMEDIATE / 'summary_equities.csv', index=False)
print(f"Saved {len(main_df_equities)} equity ETFs")

In [8]:
#display etf ticker and name grouped by region_category and intra_region_category and show all rows, sort by intra_region_category 
main_df_equities.groupby(['asset_class','region_category', 'intra_region_category'])[['ticker', 'name','country','ter','beta','yday_close_price','last_five_years','last_year_dividends','available_on_investengine','yday_close_price_date', 'yday_close_price']].last().reset_index().sort_values(by=['asset_class', 'intra_region_category'], ascending=[True, True])

,asset_class,region_category,intra_region_category,ticker,name,country,ter,beta,yday_close_price,last_five_years,last_year_dividends,available_on_investengine,yday_close_price_date,yday_close_price
0,equity,Developed_APAC,Beta,PRIJ,Amundi Prime Japan UCITS ETF DR (D),Japan,0.05,1.092259,2368.25,43.51,1.87,True,2025-05-23,2368.25
2,equity,Developed_AmericasandUK,Beta,LCUK,Amundi UK Equity All Cap UCITS ETF Dist,United Kingdom,0.04,1.030462,12.36,71.43,3.74,True,2025-05-23,12.36
4,equity,Developed_EMEA,Beta,XDDX,Xtrackers DAX ESG Screened UCITS ETF 1D,Germany,0.09,1.606825,12650.0,95.61,2.81,True,2025-05-23,12650.0
1,equity,Developed_APAC,High Yield,AUAD,UBS ETF (IE) MSCI Australia UCITS ETF (AUD) A-dis,Australia,0.40,NaN,1827.0,71.76,3.58,True,2025-05-23,1827.0
3,equity,Developed_AmericasandUK,High Yield,QYLP,Global X Nasdaq 100 Covered Call UCITS ETF D,United States,0.45,NaN,11.45,NaN,11.72,True,2025-05-23,11.45
5,equity,Developed_EMEA,High Yield,IMIB,iShares FTSE MIB UCITS ETF EUR (Dist),Italy,0.35,NaN,2012.75,157.90,4.33,True,2025-05-23,2012.75
6,equity,Emerging_APACandEMEA,High Yield,HMCH,HSBC MSCI China UCITS ETF USD,China,0.28,NaN,554.62,-8.95,2.67,True,2025-05-23,554.62
7,equity,Emerging_Americas,High Yield,IBZL,iShares MSCI Brazil UCITS ETF (Dist),Brazil,0.74,NaN,1640.0,55.80,5.59,True,2025-05-23,1640.0


In [ ]:
############# BONDS #############
main_df_bonds = pd.DataFrame()

# Manually curated bond ETF list
bonds_data = {
    'asset_class': ['bonds'] * 8,
    'region_category': [
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_EMEA', 'Developed_EMEA', 'Developed_EMEA',
        'Emerging_APACandEMEA',
    ],
    'intra_region_category': ['Govt', 'Corp', 'Govt', 'Corp', 'Govt', 'Corp', 'High yield', 'Corp'],
    'ticker': ['IGLT', 'SLXX', 'TRXG', 'UC81', 'PRIR', 'VECP', 'JNKE', 'EMCP'],
}
df_bonds_list = pd.DataFrame(bonds_data)
df_bonds_list = df_bonds_list[df_bonds_list['ticker'] != 'JNKE']  # not on InvestEngine

for filepath in sorted(DATA_RAW.glob("justetf_class-bonds_*.csv")):
    csvl = filepath.name
    try:
        asset_class = get_asset_class_from_filename(csvl)
        region_category = get_region_category_from_filename(csvl)
        print(f'Processing {asset_class} file for {region_category}: {csvl}')

        try:
            csv_df = pd.read_csv(filepath)
            if csv_df.empty:
                print(f'Empty file: {csvl}')
                continue
        except pd.errors.EmptyDataError:
            print(f'Empty file: {csvl}')
            continue

        csv_df['asset_class'] = asset_class
        csv_df['region'] = csv_df['region'].fillna('N/A')
        csv_df['country'] = csv_df['country'].fillna('N/A')

        for _, row in df_bonds_list.iterrows():
            if row['ticker'] in csv_df['ticker'].values:
                ticker = row['ticker']
                print(f'  Found {ticker} in {csvl}')
                csv_df_ticker = csv_df[csv_df['ticker'] == ticker].copy()
                csv_df_ticker['intra_region_category'] = row['intra_region_category']
                csv_df_ticker['region_category'] = row['region_category']
                main_df_bonds = pd.concat([main_df_bonds, csv_df_ticker], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')

main_df_bonds['available_on_investengine'] = (
    main_df_bonds['ticker'].apply(check_etf_availability)
)

main_df_bonds[['yday_close_price_date', 'yday_close_price']] = (
    main_df_bonds['ticker'].apply(_get_price).to_list()
)

main_df_bonds.to_csv(DATA_INTERMEDIATE / 'summary_bonds.csv', index=False)
print(f"Saved {len(main_df_bonds)} bond ETFs")

In [ ]:
summary_df = pd.concat([main_df_equities, main_df_bonds], ignore_index=True)
summary_df.to_csv(DATA_INTERMEDIATE / 'summary_all.csv', index=False)
print(f"Saved {len(summary_df)} total ETFs to summary_all.csv")
summary_df